# 07 — Feature Relationships
## Guided EDA examples

This notebook uses synthetic student-performance data to explore feature-to-target and feature-to-feature relationships. No external data download is required.

**Reminder:** association is not causation, and correlation alone does not determine which features should be retained in a model.


## 1. Imports and reproducible dataset

In [ ]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

rng = np.random.default_rng(17)
n = 220

program = rng.choice(["Data Science", "Cybersecurity", "Software Engineering"], n)
study_group = rng.choice(["Low", "Medium", "High"], n, p=[0.25, 0.50, 0.25])
study_hours = np.clip(rng.normal(8, 3, n), 0, 20)
attendance_pct = np.clip(rng.normal(82, 9, n), 45, 100)
prior_score = np.clip(rng.normal(68, 12, n), 20, 99)
program_effect = pd.Series(program).map({
    "Data Science": 2.0, "Cybersecurity": 0.0, "Software Engineering": 1.0
}).to_numpy()
group_effect = pd.Series(study_group).map({"Low": -4.0, "Medium": 0.0, "High": 4.0}).to_numpy()
noise = rng.normal(0, 7, n)

exam_score = np.clip(
    16 + 1.7 * study_hours + 0.20 * attendance_pct + 0.30 * prior_score
    + program_effect + group_effect + noise,
    0, 100
)

df = pd.DataFrame({
    "student_id": [f"S{i:03d}" for i in range(1, n + 1)],
    "program": program,
    "study_group": study_group,
    "study_hours": study_hours.round(1),
    "attendance_pct": attendance_pct.round(1),
    "prior_score": prior_score.round(1),
    "exam_score": exam_score.round(1)
})
df["result_band"] = pd.cut(
    df["exam_score"], bins=[-0.1, 49, 69, 84, 100],
    labels=["Needs support", "Developing", "Proficient", "Advanced"]
)
df.loc[rng.choice(df.index, 7, replace=False), "study_hours"] = np.nan
df.loc[rng.choice(df.index, 5, replace=False), "prior_score"] = np.nan
df.head()


## 2. Inspect schema, grain, and missingness

In [ ]:
print("Shape:", df.shape)
print(df.dtypes)
print("Duplicate rows:", int(df.duplicated().sum()))
print("Duplicate student IDs:", int(df["student_id"].duplicated().sum()))
quality = pd.DataFrame({
    "dtype": df.dtypes.astype(str),
    "missing_count": df.isna().sum(),
    "missing_percent": (df.isna().mean() * 100).round(2),
    "distinct_count": df.nunique(dropna=False)
})
display(quality)
display(df.sample(8, random_state=12))


## 3. Feature-to-target correlations

In [ ]:
numeric = df.select_dtypes(include="number").drop(columns=["student_id"], errors="ignore")
target_corr = numeric.corr(method="spearman")["exam_score"].drop("exam_score")
display(target_corr.reindex(target_corr.abs().sort_values(ascending=False).index).to_frame("spearman_corr"))


## 4. Feature-to-feature correlation

In [ ]:
pearson_corr = numeric.corr(method="pearson")
spearman_corr = numeric.corr(method="spearman")
display(pearson_corr.round(2))
display(spearman_corr.round(2))


## 5. Heatmap

In [ ]:
plt.figure(figsize=(8, 6))
sns.heatmap(pearson_corr, annot=True, fmt=".2f", cmap="coolwarm", vmin=-1, vmax=1, center=0)
plt.title("Pearson correlation matrix")
plt.tight_layout()
plt.show()


## 6. Unique correlation pairs

In [ ]:
upper = pearson_corr.where(np.triu(np.ones(pearson_corr.shape), k=1).astype(bool))
pairs = upper.stack().rename("correlation").reset_index()
pairs.columns = ["feature_a", "feature_b", "correlation"]
pairs["abs_correlation"] = pairs["correlation"].abs()
display(pairs.sort_values("abs_correlation", ascending=False))
display(pairs[pairs["abs_correlation"] > 0.75])


## 7. Scatter plot: two numeric features and a category

In [ ]:
sns.scatterplot(data=df, x="study_hours", y="exam_score", hue="program", alpha=0.75)
plt.title("Study hours and exam score by program")
plt.xlabel("Study hours per week")
plt.ylabel("Exam score")
plt.tight_layout()
plt.show()


## 8. Pair plot

In [ ]:
sns.pairplot(
    df[["study_hours", "attendance_pct", "prior_score", "exam_score", "program"]],
    hue="program", corner=True, plot_kws={"alpha": 0.65, "s": 24}
)
plt.show()


## 9. Numeric feature across categories

In [ ]:
sns.boxplot(data=df, x="program", y="exam_score")
plt.title("Exam-score distribution by program")
plt.xticks(rotation=20)
plt.tight_layout()
plt.show()


## 10. Group summaries and pivot table

In [ ]:
group_summary = df.groupby(["program", "study_group"], observed=False)["exam_score"].agg(
    count="count", mean="mean", median="median", std="std"
).round(2)
display(group_summary)
display(df.pivot_table(
    values="exam_score", index="program", columns="study_group",
    aggfunc="mean", observed=False
).round(1))


## 11. Categorical relationships

In [ ]:
counts = pd.crosstab(df["program"], df["result_band"], dropna=False)
row_pct = pd.crosstab(
    df["program"], df["result_band"], normalize="index", dropna=False
).mul(100).round(1)
display(counts)
display(row_pct)


## 12. Relationship by subgroup

In [ ]:
rows = []
for name, group in df.groupby("program", observed=False):
    valid = group[["study_hours", "exam_score"]].dropna()
    rows.append({"program": name, "n": len(valid), "pearson_r": valid["study_hours"].corr(valid["exam_score"])})
display(pd.DataFrame(rows).round(3))
g = sns.relplot(data=df, x="study_hours", y="exam_score", col="program", kind="scatter", alpha=0.7)
g.set_axis_labels("Study hours per week", "Exam score")
plt.show()


## 13. Interpretation prompts

1. Which features have the strongest observed association with the target?
2. Which input features appear to overlap with one another?
3. Do the study-hours patterns appear similar across programs?
4. Are any groups small or affected by missing data?
5. What potential confounders or leakage risks would you investigate?
6. Which findings are descriptive rather than causal?

## Key takeaways
Use correlation as one lens, not a decision rule. Pair it with visual inspection, domain knowledge, data-quality checks, and appropriate model validation.
